### Middleware
###### middleware provides a way to more tightly control what happens inside the agent
###### it is a function that takes in a list of messages and returns a list of messages
###### it can be used to add logging, add a wait time, or add a custom function
###### it can be used to transform prompts , tool selection and output formatting
###### adding retry logic, rate limiting, and early termination logic 
###### apply guardrails, pii detection, and other policies


In [13]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROK_API_KEY")

###### builtin middlewares: - summarization , human in the feedback , model calling limit

#### summarization middleware 
###### automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older contexts , Summarization is useful for the following :


 - long running conversations that exceed context winodows
 - multi - turn dialouges with extensive historu
 - applications where preserving full conversation context matters

 ###### context window is the llms working memory , the maximum amount of text measured in tokens that an llm can process and reference at a time.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### message based summarization
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages", 10),#can be tokens 
            keep=("messages", 4),
        )
    ],
    system_prompt=SystemMessage(
        content="You are a helpful assistant that can summarize conversations."
    ),
)

In [ ]:
### Run with thread id
config ={"configurable":{"thread_id":"123"}} #SAME CONVERSATION ACROSS TURNS 

In [ ]:
#alternative test data
questions=[
    "what is 2+2?",
    "what is 5*8?",
    "what is 100-3?",
    "what is 64/2?",
    "what is 10%2?",
    "what is 10**2?", 
    "what is 10//4?",
    "what is 10%3?",
]

for question in questions:
    response = agent.invoke({"messages":[HumanMessage(content=question)]},config=config)
    print(f"message: {response}")
    print(f"message:: {len(response['messages'])}")


message: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='95cb5c6c-8dd0-4c56-9b30-6c3de82d1b1e'), AIMessage(content="2 + 2 = 4. \n\nOur conversation so far: We've discussed a simple math problem, and I've provided the answer to 2 + 2, which is 4.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 52, 'total_tokens': 93, 'completion_time': 0.152276657, 'completion_tokens_details': None, 'prompt_time': 0.003139385, 'prompt_tokens_details': None, 'queue_time': 0.052641825, 'total_time': 0.155416042}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff167-b501-7fe3-bf18-4cdeb3ecf4e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 41, 'total_tokens': 93}), HumanMessage(content='what is 2+2?', additional_

#### token size trigger- present state token , fraction- based on context of llm

In [20]:
#### token size trigger- present state token , fraction- based on context of llm
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

def search_hotels(city:str)->str:
    """search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star , $350/night , spa , pool , gym
    2.City inn - 4 star , $180/night , business center
    3.Budget stay - 3 star, $65/night , free wifi"""

### token based summarization
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("tokens", 500),#can be tokens 
            keep=("tokens", 200),
        )
    ],
    system_prompt=SystemMessage(
        content="You are a helpful assistant that can summarize conversations."
    ),
)

### Run with thread id
config ={"configurable":{"thread_id":"12"}} 


In [ ]:
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4

cities = ["paris", "london", " tokyo", "hyderabad", "mumbai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotel in {city}")]},config=config
    )
    tokens= count_tokens(response["messages"])
    print(f"{city}:  ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

paris:  ~137 tokens, 4 messages
[HumanMessage(content='find hotel in paris', additional_kwargs={}, response_metadata={}, id='88dcccc1-c6fc-4f09-9290-2e043c57c26e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2dqfb0k0g', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 236, 'total_tokens': 251, 'completion_time': 0.038934267, 'completion_tokens_details': None, 'prompt_time': 0.012765773, 'prompt_tokens_details': None, 'queue_time': 0.161764475, 'total_time': 0.05170004}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff19b-ef8d-7b61-8845-586bed84be2c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2dqfb0k0g', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

### human in the loop
##### pauses agent execution for human approval , editing or rejection of tools calls before they execute. confirmation in case of critical tasks.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool